In [ ]:
import LanusStats as ls
import matplotlib.pyplot as plt
import ternary
import pandas as pd
from scipy import stats

In [ ]:
fbref = ls.Fbref()

In [ ]:
ls.get_available_leagues('Fbref')

In [ ]:
south_american_leagues = [
    'Copa de la Liga',
    'Primera Division Argentina',
    'Primera Division Uruguay',
    'Brasileirao',
    'Brasileirao B',
    'Primera Division Colombia',
    'Primera Division Chile',
    'Primera Division Peru',
    'Primera Division Venezuela',
    'Primera Division Ecuador',
    'Primera Division Bolivia',
    'Primera Division Paraguay',
    'Brasileirao F',
]
season = '2024'

league_data = {
    league: fbref.get_all_player_season_stats(league, season)
    for league in south_american_leagues
}
df = pd.concat(
    [data.assign(competition=league) for league, data in league_data.items()],
    ignore_index=True,
)
df.head()

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
list(df.columns)

In [ ]:
stat1 = 'passing_PrgP'
stat2 = 'stats_PrgC'
stat3 = 'gca_SCA90'

In [ ]:
df[['stats_90s']].describe()

In [ ]:
df = df[
    (df['stats_90s'] > 10)
    & df['stats_Pos'].fillna('').str.contains('DF', regex=False)
].reset_index(drop=True)

In [ ]:
df.groupby('competition', sort=False).size().sort_values(ascending=False)

In [ ]:
competition_plots = {}

for competition, group in df.groupby('competition', sort=False):
    group = group.dropna(subset=[stat1, stat2, stat3])
    if group.empty:
        print(f'{competition}: no hay defensores con datos suficientes para graficar.')
        continue

    points = []
    colors = []
    for _, player in group.iterrows():
        percentiles = [
            stats.percentileofscore(group[column], player[column])
            for column in (stat1, stat2, stat3)
        ]
        total = sum(percentiles)
        if total == 0:
            raise ValueError(f'No se pueden normalizar los percentiles para {competition}.')
        points.append(tuple(value / total * 100 for value in percentiles))
        colors.append(total)

    competition_plots[competition] = (points, colors)

In [ ]:
if not competition_plots:
    raise ValueError('No hay competiciones con jugadores que cumplan los filtros.')

for competition, (points, colors) in competition_plots.items():
    mult = 100 / 3
    fig, tax = ternary.figure(scale=100)
    fig.set_size_inches(9, 9)
    tax.boundary(linewidth=2.0)
    tax.gridlines(multiple=mult, color='#4A2E19')

    color_max = max(colors)
    if min(colors) == color_max:
        color_max += 1
    tax.scatter(
        points,
        edgecolors='darkgrey',
        linewidths=.25,
        c=colors,
        cmap='OrRd',
        vmin=min(colors),
        vmax=color_max,
    )
    tax.set_title(
        f'Gr?fico ternario - {competition} ({season})',
        fontsize=14,
        color='#4A2E19',
        pad=20,
    )
    tax.right_corner_label(stat1, fontsize=12, offset=.2)
    tax.top_corner_label(stat2, fontsize=12, offset=.15)
    tax.left_corner_label(stat3, fontsize=12, offset=.2)
    tax.left_axis_label('Defensores y acciones ofensivas', fontsize=9, offset=.05, color='#cac9c5')
    tax.right_axis_label('Acciones progresivas', fontsize=9, offset=.05, color='#cac9c5')
    tax.bottom_axis_label('Percentiles normalizados', fontsize=7, offset=.07, color='#4A2E19')
    tax.ticks(axis='lbr', linewidth=1, multiple=20)
    tax.clear_matplotlib_ticks()
    tax.get_axes().axis('off')
    plt.show()